## 1. 필요한 라이브러리 불러오기

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import time

## 2. 하이퍼 파라미터 및 gpu 사용

In [ ]:
# Hyper Parameters
IMAGE_SIZE = 224
LEARNING_RATE = 0.01
epochs = 20
batch_size = 128

# GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(torch.__version__)
print(device)

2.9.0+cu126
cuda


## 3. 데이터 준비

In [3]:
# 데이터 준비
cifar_mean = [0.4914, 0.4822, 0.4465]
cifar_std = [0.2023, 0.1994, 0.2010]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std)
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std)
])

train_data = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_data = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:03<00:00, 47.3MB/s]


In [4]:
# 데이터 형상 확인
images, labels = next(iter(train_dataloader))
print(images[0].shape)

torch.Size([3, 224, 224])


## 4. Block 단위 클래스

In [ ]:
# block 단위 클래스 정의
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU()
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(planes)
        )
    
        # Projection Shortcut or Identity mapping
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            # 채널 수와 해상도를 일치시킨다.
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        # F(x)
        f_x = self.conv1(x)
        f_x = self.conv2(f_x)
        
        # F(x) + x
        shortcut_x = self.shortcut(x)
        out = f_x + shortcut_x

        # ReLU(F(x) + x)
        out = F.relu(out)
        return out

## 5. ResNet 클래스

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()
        
        # Stem (입력부)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(), 
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # Layers (층)
        self.layers = nn.Sequential(
            BasicBlock(64, 64, stride=1),
            BasicBlock(64, 64, stride=1),
            BasicBlock(64, 128, stride=2),
            BasicBlock(128, 128, stride=1),
            BasicBlock(128, 256, stride=2),
            BasicBlock(256, 256, stride=1),
            BasicBlock(256, 512, stride=2),
            BasicBlock(512, 512, stride=1)
        ) 
    
        # Head (출력부)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)

    def forward(self, x):
        out = self.stem(x)
    
        out = self.layers(out)
        
        out = self.avgpool(out) 
        out = torch.flatten(out, 1) # flatten
        out = self.fc(out)
        return out

## 6. 훈련 준비 및 optimizer 선언

In [ ]:
model = ResNet()
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

## 7. 훈련

In [ ]:
from google.colab import drive

# 구글 마운트 및 경로 설정
drive.mount('/content/drive')
save_dir = '/content/drive/{원하는경로}'
os.makedirs(save_dir, exist_ok=True)

print(f"학습 시작 (Device: {device})")
best_acc = 0.0

for epoch in range(epochs):
    start_time = time.time()

    # Training
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Validation 
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total
    epoch_loss = running_loss / len(train_dataloader)
  
    optimizer.step()
    current_lr = optimizer.param_groups[0]['lr']

    # Best Model 저장
    if epoch_acc > best_acc:
        best_acc = epoch_acc
        
        save_path = os.path.join(save_dir, "resnet18_best_model.pth")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'acc': epoch_acc,
        }, save_path)
        print(f"모델 저장됨. (Acc: {epoch_acc:.2f}%)")

    epoch_time = time.time() - start_time
    
    print(f"Epoch [{epoch+1:02d}/{epochs}] | "
          f"Loss: {epoch_loss:.4f} | "
          f"Test Acc: {epoch_acc:.2f}% | "
          f"LR: {current_lr:.4f} | "
          f"Time: {epoch_time:.1f}s")

print(f"\n학습 종료{best_acc:.2f}%")